# Leak-auditor fine-tune, run 5: Qwen3.5 9B, three seeds

Everything happens here, on the GPU. Nothing in this folder knows test 5's answers: the notebook only writes
answers, and their fingerprints.

1. A quick check of answering 16 prompts at a time against one at a time (it is recorded; only if batched answers come
   back unreadable does it answer one prompt at a time, for every model).
2. The **untrained** model answers every test prompt 3 times (the 42 generated tables, real test 5, and real test 5
   disguised; at most 10 columns per prompt), thinking off.
3. **Three training runs**, identical but for the seed (3407, 3408, 3409): a LoRA add-on, 2 passes over
   `train.jsonl`, the best checkpoint (lowest validation loss) kept. After each, that model answers the same prompts 3
   times.
4. The run with the lowest validation loss is saved as two GGUF files (Q4_K_M and Q6_K, the laptop sizes), and each
   answers every prompt once with llama.cpp (greedy).
5. `answers_manifest_run5.json`: the SHA-256 of every answer file.

**Before you start:** pick an A100 80 GB (or an H100) pod with RunPod's PyTorch 2.8.0 template, container disk
200 GB, volume disk 200 GB. Upload `train.jsonl`, `val.jsonl`, `test_prompts.jsonl` and this notebook into the folder
JupyterLab opened in (`/workspace`). Then **Run > Run All Cells**. It takes about 8-10 hours on an A100 (less on an
H100); you can close the browser tab, it keeps running.

**If the pod restarts or a cell stops:** open the notebook again and **Run All Cells**. Finished steps are kept (answer
files, finished training runs); a training run that was cut off starts again.

**At the end:** download `run5_results.zip` (right-click > Download) and, if you want the model for LM Studio, the
`...Q4_K_M.gguf` file. Then **terminate the pod** in the RunPod console so billing stops.

If a cell stops with an error, send Claude a screenshot of it.

In [ ]:
# ---- settings (run 5, as pre-registered: change nothing) ----
import os
# Qwen3.5 mixes in linear-attention layers. Unsloth's automatic packing for them needs GPU kernels RunPod's PyTorch
# template lacks, and then stops on the first training step; this makes it train on ordinary padded batches.
os.environ["UNSLOTH_EXPERIMENTAL_HYBRID_PACKING"] = "0"
os.environ["UNSLOTH_BLOCK_COMPILE"] = "0"
MODEL_NAME = "unsloth/Qwen3.5-9B"   # the same model as qwen/qwen3.5-9b in LM Studio
TRAIN_FILE = "train.jsonl"
VAL_FILE = "val.jsonl"
TEST_FILE = "test_prompts.jsonl"    # every test prompt, exactly as Claude and GPT saw theirs; no answer key
SEEDS = [3407, 3408, 3409]          # three training runs, identical but for the seed
BASE_FILE = "answers_base_run5.jsonl"
SEED_FILE = "answers_s%d_run5.jsonl"
GGUF_FILE = "answers_gguf_%s_run5.jsonl"
SEED_RECORD = "seed_s%d_run5.json"  # one per finished training run: its losses, best checkpoint and session
RUN_FILE = "train_run5.json"
MANIFEST_FILE = "answers_manifest_run5.json"
CHECK_FILE = "batching_check_run5.json"
RESULTS_ZIP = "run5_results.zip"
MAX_SEQ_LENGTH = 16384              # longest prompt + answer the model handles
TRAIN_MAX_TOKENS = 8192             # longest training example allowed (longer ones are left out)
EPOCHS = 2
LEARNING_RATE = 1e-4
LR_SCHEDULER = "cosine"
WARMUP_RATIO = 0.03                 # the first 3% of the updates warm up
LORA_RANK = 32                      # LoRA alpha = rank
BATCH_SIZE = 4                      # examples per step
GRAD_ACCUM = 4                      # 4 x 4 = 16 examples per update
EVAL_STEPS = 100                    # validation loss every 100 updates; a LoRA checkpoint is saved at each
SAVE_TOTAL_LIMIT = 3                # checkpoints kept (the best one always among them); the best is loaded at the end
K_SAMPLES = 3                       # answers per prompt (sampled together); a column's score is their mean
ANSWER_BATCH = 16                   # prompts answered at a time (sorted by length, left padding), for every model
BATCH_CHECK_N = 16                  # prompts in the batching check
EVAL_MAX_NEW_TOKENS = 2048          # cap on one answer (a 10-column answer is about 350-450 tokens)
EVAL_TEMPERATURE, EVAL_TOP_P, EVAL_TOP_K = 0.6, 0.95, 20   # the sampling of every earlier run
GEN_SEED = 20261101                 # the sampling seed, the same for every model
REMINDER_ALL = "Reminder: answer with one json object only, in exactly the form above, covering all %d columns."
REMINDER_MISSING = ("Reminder: your previous answer left out these columns: %s. Answer with one json object in the "
                    "same form, covering just these columns, with the names exactly as given.")
OUTPUT_NAME = "leakaudit5-qwen3.5-9b"
GGUF_QUANTS = ["q4_k_m", "q6_k"]    # the two laptop sizes, made from the run with the lowest validation loss
LLAMA_SLOTS = 6                     # prompts llama.cpp answers at once (each in its own slot)
LLAMA_CTX_PER_SLOT = 8192
LLAMA_PORT = 8089

In [ ]:
# ---- checks: GPU, files, packages ----
import os, sys, json, time, glob, hashlib, subprocess, importlib.util, importlib.metadata, re, gc, shutil, zipfile
for p in (TRAIN_FILE, VAL_FILE, TEST_FILE):
    assert os.path.exists(p), "Upload %s into this folder (%s) first." % (p, os.getcwd())

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if importlib.util.find_spec("unsloth") is None:
    # RunPod's PyTorch template has no Unsloth: install the set Unsloth's own Qwen3.5 notebook uses (5-10 minutes)
    print("installing Unsloth (5-10 minutes) ...")
    tv = importlib.metadata.version("torch") if importlib.util.find_spec("torch") else ""
    pip("--upgrade", "unsloth", "unsloth_zoo", *(["xformers==0.0.32.post2"] if tv.startswith("2.8") else []))
    pip("--no-deps", "--upgrade", "tokenizers>=0.22.0,<=0.23.0", "trl==0.22.2")
    pip("transformers==5.2.0")          # Qwen3.5 needs transformers 5
    pip("--no-deps", "--upgrade", "torchao>=0.16.0")
    print("installed")
import unsloth  # noqa: F401  (Unsloth must be imported before torch, transformers and trl)
import torch
assert torch.cuda.is_available(), "No GPU visible: this pod needs an NVIDIA GPU (an A100 80 GB or an H100)."
gpu = torch.cuda.get_device_properties(0)
print("GPU: %s, %.0f GB" % (gpu.name, gpu.total_memory / 1024**3))
if gpu.total_memory / 1024**3 < 40:
    print("WARNING: under 40 GB of GPU memory. Use an A100 80 GB or an H100.")
free_gb = shutil.disk_usage(".").free / 1024**3
print("free disk here: %.0f GB" % free_gb)
if free_gb < 100:
    print("WARNING: under 100 GB free here; the GGUF step needs about 60 GB. Give the pod more volume disk.")

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def read_json(p):
    try:
        with open(p) as f:
            return json.load(f)
    except (OSError, ValueError):
        return None

def write_json(p, obj):
    tmp = p + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, p)

RUN = read_json(RUN_FILE) or {}
files_now = {p: sha256(p) for p in (TRAIN_FILE, VAL_FILE, TEST_FILE)}
if RUN.get("files") and RUN["files"] != files_now:
    raise SystemExit("These files differ from the ones this run started with (%s): put the original ones back, or "
                     "tell Claude." % ", ".join(p for p in files_now if RUN["files"].get(p) != files_now[p]))
RUN.update(run=5, gpu=gpu.name, model=RUN.get("model", MODEL_NAME), files=files_now)
if RUN.get("model_locked"):
    MODEL_NAME = RUN["model"]       # the base model this run started with (a restart loads the same one)
RUN.setdefault("started", time.strftime("%Y-%m-%d %H:%M:%S"))
RUN.setdefault("minutes", {})
RUN["test_prompts_sha256"] = files_now[TEST_FILE]
import transformers, trl
RUN["versions"] = {"torch": torch.__version__, "transformers": transformers.__version__, "trl": trl.__version__,
                   "unsloth": getattr(unsloth, "__version__", "?"), "python": sys.version.split()[0]}
print(RUN["versions"])
with open(TEST_FILE, encoding="utf-8") as f:
    TEST_ROWS = [json.loads(line) for line in f if line.strip()]
assert not any("leak_columns" in r or "probable_leak_columns" in r for r in TEST_ROWS), \
    "test_prompts.jsonl holds answer-key fields: it must be run 5's file (t4_make_test_prompts.py)"
PARTS = {}
for r in TEST_ROWS:
    PARTS.setdefault((r["table"], r["condition"]), {})[r["part"]] = r["parts"]
broken = [k for k, ps in PARTS.items() if sorted(ps) != list(range(1, max(ps.values()) + 1)) or len(set(ps.values())) != 1]
assert TEST_ROWS and not broken, "test_prompts.jsonl is incomplete (%s): make it again with t4_make_test_prompts.py" % broken[:5]
assert len({r["id"] for r in TEST_ROWS}) == len(TEST_ROWS), "test_prompts.jsonl repeats a prompt: make it again"
groups = {}
for r in TEST_ROWS:
    g = groups.setdefault(r["group"], [set(), 0])
    g[0].add(r["table"])
    g[1] += 1
print("test prompts: %d -- %s" % (len(TEST_ROWS), "; ".join("%s %d tables / %d prompts" % (g, len(v[0]), v[1])
                                                         for g, v in groups.items())))
RUN["test"] = {"prompts": len(TEST_ROWS), "groups": {g: {"tables": len(v[0]), "prompts": v[1]} for g, v in groups.items()}}
ROW_BY_ID = {r["id"]: r for r in TEST_ROWS}
write_json(RUN_FILE, RUN)

In [ ]:
# ---- helpers: load a model, the prompt (thinking off), read answers, answer every test prompt ----
from unsloth import FastLanguageModel

def load_model(name):
    return FastLanguageModel.from_pretrained(
        model_name=name,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,      # Unsloth advises against 4-bit training for Qwen3.5
        load_in_16bit=True,
        full_finetuning=False,
    )

def load_any():
    # the base model; the first load decides which copy (Unsloth's, else Qwen's), and every later load uses that one
    global MODEL_NAME
    if RUN.get("model_locked"):
        return load_model(MODEL_NAME)
    try:
        res = load_model(MODEL_NAME)
    except (OSError, ValueError) as e:   # if Unsloth's copy can't be fetched, the same weights from Qwen
        print("could not load %s (%s); trying Qwen/Qwen3.5-9B" % (MODEL_NAME, e.__class__.__name__))
        MODEL_NAME = "Qwen/Qwen3.5-9B"
        res = load_model(MODEL_NAME)
    RUN["model"], RUN["model_locked"] = MODEL_NAME, True
    write_json(RUN_FILE, RUN)
    return res

def prompt_sig(chat):
    r = TEST_ROWS[0]
    return hashlib.sha256(prompt_text(chat, r["system"], r["user"]).encode("utf-8")).hexdigest()

def check_prompt(chat, where):
    # every model must see the test prompts in exactly the untrained model's format (chat template, thinking off)
    sig = prompt_sig(chat)
    if RUN.get("prompt_sig") is None:
        RUN["prompt_sig"] = sig
        write_json(RUN_FILE, RUN)
    elif sig != RUN["prompt_sig"]:
        raise SystemExit("The prompt format of %s differs from the untrained model's (its chat template): stop here "
                         "and tell Claude." % where)

def text_parts(tokenizer):
    tok = getattr(tokenizer, "tokenizer", tokenizer)   # Qwen3.5 can come with a processor; the text tokenizer is inside
    chat = tok if getattr(tok, "chat_template", None) else tokenizer   # whichever holds Qwen3.5's chat template
    return tok, chat

def prompt_text(chat, system, user):
    # The test prompt, thinking off. Training texts are this plus the answer, so training and testing match exactly.
    return chat.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": user}],
                                    tokenize=False, add_generation_prompt=True, enable_thinking=False)

def _block_end(t, start):
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "{[":
            depth += 1
        elif ch in "}]":
            depth -= 1
            if depth == 0:
                return i
    return None

def read_answer(text):
    # The answer's json object (after any thinking), tolerating prose, code fences and trailing commas; else None.
    body = (text or "").split("</think>")[-1]
    for m in re.finditer(r"\{", body):
        end = _block_end(body, m.start())
        if end is None:
            continue
        chunk = body[m.start():end + 1]
        for cand in (chunk, re.sub(r",\s*([}\]])", r"\1", chunk)):
            try:
                obj = json.loads(cand)
            except ValueError:
                continue
            if isinstance(obj, dict) and "columns" in obj:
                return obj
    return None

def loose(name):
    return re.sub(r"(?<!\d)0+(?=\d)", "", re.sub(r"[^a-z0-9]", "", str(name).lower()))

def risks_of(obj, columns):
    # {column: risk} for this prompt's columns in one answer object, matched as step 3's reader (t5) matches: the
    # exact name first, else a case- and punctuation-blind match that only one of the prompt's columns has
    cols = obj.get("columns") if isinstance(obj, dict) else None
    if isinstance(cols, dict):
        cols = [dict(v, column=k) if isinstance(v, dict) else {"column": k, "leak_risk": v} for k, v in cols.items()]
    keys = {}
    for c in columns:
        keys.setdefault(loose(c), []).append(c)
    unique = {k: v[0] for k, v in keys.items() if len(v) == 1}
    out = {}
    for e in cols if isinstance(cols, list) else []:
        if not isinstance(e, dict) or e.get("column", e.get("name")) is None:
            continue
        name = str(e.get("column", e.get("name"))).strip().strip("`'\"")
        hit = name if name in columns else unique.get(loose(name))
        if hit is None or hit in out:
            continue
        try:
            out[hit] = float(e.get("leak_risk"))
        except (TypeError, ValueError):
            out[hit] = float("nan")
    return out

def rated_cols(got):
    # the columns of one answer that got a usable (numeric) risk
    return {c for c, v in got.items() if v == v}

def answer_risks(rec, columns):
    # {column: risk}: the mean over the first-round samples that gave the column a numeric risk, or, if none did,
    # over the retry samples (t5_import_gpu.py does the same)
    texts, retry = rec.get("texts") or [], rec.get("texts_retry") or []
    first = [risks_of(read_answer(t), columns) for t in texts]
    again = [risks_of(read_answer(t), columns) for t in retry]
    out = {}
    for c in columns:
        vals = [g[c] for g in first if c in rated_cols(g)] or [g[c] for g in again if c in rated_cols(g)]
        if vals:
            out[c] = sum(vals) / len(vals)
    return out

def read_done(out_file):
    # (answers by prompt id, whether the file held NUL bytes or broken lines)
    done, nul, bad = {}, 0, 0
    if os.path.exists(out_file):
        with open(out_file, "rb") as f:
            raw = f.read()
        nul = raw.count(b"\x00")
        for line in raw.replace(b"\x00", b"").decode("utf-8", "replace").split("\n"):
            if not line.strip():
                continue
            try:
                r = json.loads(line)
                if not isinstance(r.get("texts"), list):
                    raise ValueError("not an answer line")
                done[r["id"]] = r
            except (ValueError, KeyError, TypeError, AttributeError):
                bad += 1
        if nul or bad:
            print("  %s: %d NUL bytes and %d unreadable lines skipped" % (out_file, nul, bad))
    return done, bool(nul or bad)

def write_all(out_file, done):
    # The whole file, rewritten after every batch: written aside, synced to disk, then swapped in (never appended to:
    # run 2's file came back from RunPod's volume with NUL bytes in front after append-mode writes).
    tmp = out_file + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in done.values():
            f.write(json.dumps(r) + "\n")   # ASCII only: no character inside a line can split it
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, out_file)

def load_answers(out_file, session=None):
    # The answers on file, minus any to a prompt not in today's test_prompts.jsonl and, with a session, any made by
    # another model (e.g. a training run that was started again); a damaged file is written again cleanly
    done, damaged = read_done(out_file)
    stale = [k for k in done if k not in ROW_BY_ID]
    other = [k for k in done if k in ROW_BY_ID and session is not None and done[k].get("session") != session]
    if stale:
        print("  %s: %d answers to prompts not in %s dropped" % (out_file, len(stale), TEST_FILE))
    if other:
        print("  %s: %d answers from another model dropped (they are asked again)" % (out_file, len(other)))
    for k in stale + other:
        del done[k]
    if stale or other or damaged:
        write_all(out_file, done)
    return done

def complete(out_file, k, session=None):
    done = read_done(out_file)[0]
    return (len(done) == len(TEST_ROWS) and all(i in done for i in ROW_BY_ID) and
            all(len(r["texts"]) == k and (session is None or r.get("session") == session) for r in done.values()))

def verify_file(out_file, k):
    with open(out_file, "rb") as f:
        raw = f.read()
    assert b"\x00" not in raw, "%s holds NUL bytes: tell Claude" % out_file
    rows = [json.loads(x) for x in raw.decode("utf-8").split("\n") if x.strip()]
    assert len(rows) == len(TEST_ROWS) and {r["id"] for r in rows} == set(ROW_BY_ID), \
        "%s holds %d answers, expected %d: run this cell again" % (out_file, len(rows), len(TEST_ROWS))
    assert all(len(r["texts"]) == k for r in rows), "%s: an answer has the wrong number of samples" % out_file
    return len(rows)

def stop_ids(tok):
    ids = set()
    for t in ("<|im_end|>", "<|endoftext|>"):
        i = tok.convert_tokens_to_ids(t)
        if isinstance(i, int) and i >= 0 and i != getattr(tok, "unk_token_id", None):
            ids.add(i)
    for i in (tok.eos_token_id, tok.pad_token_id):
        if isinstance(i, int):
            ids.add(i)
    return ids

def gen_batch(model, tok, prompts, seed, greedy=False):
    # Several prompts in one call: left padding with an attention mask, K_SAMPLES sampled answers per prompt (one
    # greedy answer if greedy). Per prompt, a list of (text, new tokens).
    side = tok.padding_side
    tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    try:
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        torch.manual_seed(seed)
        kw = dict(max_new_tokens=EVAL_MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id)
        if greedy:
            kw.update(do_sample=False, num_return_sequences=1)
        else:
            kw.update(do_sample=True, temperature=EVAL_TEMPERATURE, top_p=EVAL_TOP_P, top_k=EVAL_TOP_K,
                      num_return_sequences=K_SAMPLES)
        out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], **kw)
    finally:
        tok.padding_side = side
    n_ret = 1 if greedy else K_SAMPLES
    plen = enc["input_ids"].shape[1]
    stops = stop_ids(tok)
    res = []
    for i in range(len(prompts)):
        one = []
        for j in range(n_ret):
            gen = out[i * n_ret + j][plen:]
            ids = [int(x) for x in (gen.tolist() if hasattr(gen, "tolist") else gen)]
            n = next((k + 1 for k, x in enumerate(ids) if x in stops), len(ids))
            text = tok.decode(ids[:n], skip_special_tokens=False)
            for stop in ("<|im_end|>", "<|endoftext|>"):
                text = text.split(stop)[0]
            one.append((text, n))
        res.append(one)
    return res

SPLITS = {"oom_splits": 0}

def gen_safe(model, tok, prompts, seed):
    # gen_batch, and if the GPU runs out of memory, the same prompts in two halves (down to one prompt at a time)
    try:
        return gen_batch(model, tok, prompts, seed)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower() or len(prompts) == 1:
            raise
    gc.collect()
    torch.cuda.empty_cache()
    SPLITS["oom_splits"] += 1
    half = len(prompts) // 2
    return gen_safe(model, tok, prompts[:half], seed) + gen_safe(model, tok, prompts[half:], seed + 7)

def batching_check(model, tok, chat):
    # The first BATCH_CHECK_N prompts, greedy (one answer each), in one batch and one at a time. Recorded; the rule
    # (fixed in advance): answers go one prompt at a time only if 2 or more answers are readable in one mode and not
    # in the other (or batching raises). Different numbers in the two modes are expected (bf16 arithmetic differs
    # with the batch shape; in run 4 the two modes agreed on 87% of the flags, as often as two samples of the same
    # model, 89%).
    FastLanguageModel.for_inference(model)
    rows = TEST_ROWS[:BATCH_CHECK_N]
    prompts = [prompt_text(chat, r["system"], r["user"]) for r in rows]
    t0 = time.time()
    single = [gen_batch(model, tok, [p], GEN_SEED, greedy=True)[0][0][0] for p in prompts]
    res = {"prompts": len(rows), "batch_tried": ANSWER_BATCH, "seconds_one_at_a_time": round(time.time() - t0, 1)}
    t1 = time.time()
    try:
        batched = [x[0][0] for x in gen_batch(model, tok, prompts, GEN_SEED, greedy=True)]
    except Exception as e:   # e.g. out of memory, or a model that cannot take padded batches
        gc.collect()
        torch.cuda.empty_cache()
        res.update(ok=False, answer_batch=1, why="batched generation raised %s: %s"
                   % (e.__class__.__name__, " ".join(str(e).split())[:300]))
        return res
    res["seconds_batched"] = round(time.time() - t1, 1)
    mismatch, n_cols, same = [], 0, 0
    for r, a, b in zip(rows, single, batched):
        ra, rb = risks_of(read_answer(a), r["columns"]), risks_of(read_answer(b), r["columns"])
        if bool(rated_cols(ra)) != bool(rated_cols(rb)):
            mismatch.append(r["id"])
        for c in set(ra) | set(rb):
            n_cols += 1
            same += c in ra and c in rb and (ra[c] >= 50) == (rb[c] >= 50)
    share = same / n_cols if n_cols else 0.0
    res.update(readable_mismatch=mismatch, rated_columns=n_cols, same_flag=same, same_share=round(share, 4),
               identical_texts=sum(1 for a, b in zip(single, batched) if a == b))
    if len(mismatch) >= 2:
        res.update(ok=False, answer_batch=1, why="%d answers readable in one mode and not the other" % len(mismatch))
    else:
        res.update(ok=True, answer_batch=ANSWER_BATCH, why="%d of %d answers identical, %.1f%% of the rated columns "
                   "the same flag" % (res["identical_texts"], len(rows), 100 * share))
    return res

def batch_setting(model, tok, chat):
    # ANSWER_BATCH for the whole run (every model): the saved result of the batching check, else the check itself
    global ANSWER_BATCH
    res = read_json(CHECK_FILE)
    if res is None:
        base = read_done(BASE_FILE)[0]
        used = {r.get("batch") for r in base.values()} - {None}
        if used and (model is None or len(base) == len(TEST_ROWS)):
            res = {"ok": min(used) > 1, "answer_batch": min(used), "why": "kept from the untrained answers on file"}
        elif model is None:
            return None
        else:
            t0 = time.time()
            res = batching_check(model, tok, chat)
            res["minutes"] = round((time.time() - t0) / 60, 1)
        write_json(CHECK_FILE, res)
    else:
        print("batching check: result kept from %s" % CHECK_FILE)
    ANSWER_BATCH = int(res["answer_batch"])
    RUN["batching_check"] = res
    RUN["answer_batch"] = ANSWER_BATCH
    print("batching check (%s): %s -> ANSWER_BATCH %d%s"
          % ("passed" if res.get("ok") else "failed", res.get("why"), ANSWER_BATCH,
             "; %s s one at a time vs %s s in one batch" % (res["seconds_one_at_a_time"], res["seconds_batched"])
             if "seconds_batched" in res else ""))
    return res

def answer_all(model, tok, chat, out_file, label, session):
    # K_SAMPLES answers per test prompt, ANSWER_BATCH prompts at a time (sorted by length), the whole file rewritten
    # after each batch; a re-run skips the prompts already answered. A prompt whose samples leave some column unrated
    # (or are all unreadable) is asked once more, with run 3's reminder.
    done = load_answers(out_file, session)
    FastLanguageModel.for_inference(model)
    todo = sorted([(i, r) for i, r in enumerate(TEST_ROWS) if r["id"] not in done], key=lambda x: len(x[1]["user"]))
    t0, new, ok_any, early = time.time(), 0, 0, False
    for b in range(0, len(todo), ANSWER_BATCH):
        part = todo[b:b + ANSWER_BATCH]
        s0 = time.time()
        outs = gen_safe(model, tok, [prompt_text(chat, r["system"], r["user"]) for _, r in part],
                        GEN_SEED + 1000 + part[0][0])
        recs, again = [], []
        for (i, r), samples in zip(part, outs):
            texts = [t for t, _ in samples]
            got = [risks_of(read_answer(t), r["columns"]) for t in texts]
            rated = {c for g in got for c in rated_cols(g)}
            n_read = sum(1 for g in got if rated_cols(g))
            rec = {"id": r["id"], "table": r["table"], "group": r["group"], "condition": r["condition"],
                   "label": label, "thinking": False, "texts": texts, "texts_retry": None,
                   "tokens": [n for _, n in samples], "seconds": None, "attempts": 1, "session": session,
                   "readable": n_read, "batch": ANSWER_BATCH}
            recs.append(rec)
            missing = [c for c in r["columns"] if c not in rated]
            if n_read == 0 or missing:
                names = [" ".join(str(c).split()) for c in missing]
                reminder = REMINDER_ALL % len(r["columns"]) if n_read == 0 else REMINDER_MISSING % ", ".join(names)
                again.append((i, r, rec, reminder))
        if again:
            outs2 = gen_safe(model, tok, [prompt_text(chat, r["system"], r["user"] + "\n\n" + rem)
                                          for _, r, _, rem in again], GEN_SEED + 2000 + again[0][0])
            for (i, r, rec, _), samples in zip(again, outs2):
                rec["texts_retry"] = [t for t, _ in samples]
                rec["tokens"] = rec["tokens"] + [n for _, n in samples]
                rec["attempts"] = 2
        sec = round((time.time() - s0) / len(part), 1)
        for (i, r), rec in zip(part, recs):
            rec["seconds"] = sec
            done[rec["id"]] = rec
            ok_any += bool(answer_risks(rec, r["columns"]))
        write_all(out_file, done)
        new += len(part)
        if not early and new >= 8:
            early = True
            if ok_any < new / 2.0:
                raise RuntimeError("only %d of the first %d answers are readable json: stop here and tell Claude"
                                   % (ok_any, new))
        if (b // ANSWER_BATCH) % max(1, 64 // ANSWER_BATCH) == 0 or len(done) == len(TEST_ROWS):
            el = time.time() - t0
            left = len(TEST_ROWS) - len(done)
            print("  %s: %d / %d prompts (%.0f min, about %.0f min left)"
                  % (label, len(done), len(TEST_ROWS), el / 60, el / max(new, 1) * left / 60))
    verify_file(out_file, K_SAMPLES)
    return done

def stage_stats(ans, name):
    # the stage's numbers for the run record (no scores: nothing here knows the answers)
    s = {"answers": len(ans), "samples": sum(len(r["texts"]) for r in ans.values()),
         "readable_samples": sum(int(r.get("readable") or 0) for r in ans.values()),
         "asked_again": sum(1 for r in ans.values() if r.get("attempts") == 2),
         "no_readable_answer": sum(1 for r in ans.values() if not answer_risks(r, ROW_BY_ID[r["id"]]["columns"])),
         "minutes": round(sum((r.get("seconds") or 0) for r in ans.values()) / 60, 1)}
    print("%s: %d prompts, %d of %d samples readable, %d asked again, %d with no readable answer (%.0f min)"
          % (name, s["answers"], s["readable_samples"], s["samples"], s["asked_again"], s["no_readable_answer"],
             s["minutes"]))
    return s

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ---- BEFORE: the batching check, then the untrained model answers every test prompt 3 times, thinking off ----
# (about 40 minutes in batches of 16)
def base_session():
    return "untrained %s %s" % (MODEL_NAME, (RUN.get("prompt_sig") or "")[:12])

if RUN.get("prompt_sig") and complete(BASE_FILE, K_SAMPLES, base_session()):   # a re-run: all here already
    print("%s already holds all %d prompts: kept" % (BASE_FILE, len(TEST_ROWS)))
    batch_setting(None, None, None)
    BASE = load_answers(BASE_FILE, base_session())
else:
    model, tokenizer = load_any()
    tok, chat = text_parts(tokenizer)
    check_prompt(chat, "the untrained model")
    print("the prompt ends with: %r" % prompt_text(chat, "S", "U")[-60:])
    batch_setting(model, tok, chat)
    BASE = answer_all(model, tok, chat, out_file=BASE_FILE, label="untrained (thinking off)", session=base_session())
    del model, tokenizer
    free_gpu()
RUN["base"] = stage_stats(BASE, "untrained answers")
write_json(RUN_FILE, RUN)

In [ ]:
# ---- training: the texts (the exact test prompt, thinking off, then the answer) and the trainer ----
import inspect, math
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def load_rows(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

TRAIN_ROWS, VAL_ROWS = load_rows(TRAIN_FILE), load_rows(VAL_FILE)
TEXTS = {}

def text_of(chat, row):
    m = row["messages"]
    return prompt_text(chat, m[0]["content"], m[1]["content"]) + m[2]["content"] + "<|im_end|>"

def training_texts(tok, chat):
    # once for all three runs: the texts, and the check that each starts with exactly the test prompt's tokens
    if TEXTS:
        return TEXTS
    first = TRAIN_ROWS[0]["messages"]
    p_ids = tok(prompt_text(chat, first[0]["content"], first[1]["content"]), add_special_tokens=False)["input_ids"]
    t_ids = tok(text_of(chat, TRAIN_ROWS[0]), add_special_tokens=False)["input_ids"]
    assert t_ids[:len(p_ids)] == p_ids, "the training text does not start with the test prompt's tokens: tell Claude"

    def texts(rows):
        out, lens, dropped, kinds = [], [], 0, {}
        for r in rows:
            t = text_of(chat, r)
            n = len(tok(t, add_special_tokens=False)["input_ids"])
            if n > TRAIN_MAX_TOKENS:
                dropped += 1
                continue
            out.append(t)
            lens.append(n)
            key = "%s %s" % (r.get("source", "?"), r.get("variant_kind", ""))
            kinds[key.strip()] = kinds.get(key.strip(), 0) + 1
        return out, lens, dropped, kinds

    tr, tr_lens, tr_drop, tr_kinds = texts(TRAIN_ROWS)
    va, va_lens, va_drop, _ = texts(VAL_ROWS)
    print("train: %d examples (%d longer than %d tokens left out); tokens min %d, median %d, max %d; %.2f million"
          % (len(tr), tr_drop, TRAIN_MAX_TOKENS, min(tr_lens), sorted(tr_lens)[len(tr_lens) // 2], max(tr_lens),
             sum(tr_lens) / 1e6))
    print("  by source and copy: %s" % tr_kinds)
    print("val:   %d examples (%d left out)" % (len(va), va_drop))
    assert tr_drop + va_drop <= 0.02 * (len(TRAIN_ROWS) + len(VAL_ROWS)), "too many examples are too long"
    assert va, "val.jsonl has no usable example: tell Claude"
    RUN["examples"] = {"train": len(tr), "val": len(va), "by_source": tr_kinds, "left_out_too_long": tr_drop + va_drop,
                       "train_tokens": int(sum(tr_lens)), "max_tokens": int(max(tr_lens))}
    TEXTS.update(train=tr, val=va, max_len=max(tr_lens))
    print("\n--- the end of one training text (the part the model learns to write starts after 'assistant') ---")
    print(tr[0][-1200:])
    return TEXTS

def make_trainer(model, tok, seed, texts, outdir):
    if os.path.isdir(outdir):   # checkpoints of a cut-off run: moved aside (kept), this one starts afresh
        os.replace(outdir, "%s_before_%s" % (outdir, time.strftime("%Y%m%d_%H%M%S")))
    updates = math.ceil(len(texts["train"]) / float(BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
    warmup = max(1, int(round(WARMUP_RATIO * updates)))   # warmup as a step count: works in every TRL version
    cfg = dict(dataset_text_field="text", per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
               gradient_accumulation_steps=GRAD_ACCUM, warmup_steps=warmup, num_train_epochs=EPOCHS,
               learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHEDULER, logging_steps=10, eval_strategy="steps",
               eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=EVAL_STEPS, save_total_limit=SAVE_TOTAL_LIMIT,
               load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
               optim="adamw_8bit", weight_decay=0.001, seed=seed, report_to="none", output_dir=outdir)
    params = inspect.signature(SFTConfig.__init__).parameters
    for name in ("max_length", "max_seq_length"):   # newer TRL calls it max_length; without it, long examples get cut
        if name in params:
            cfg[name] = TRAIN_MAX_TOKENS
            break
    if "padding_free" in params:   # ordinary padded batches (see the settings cell)
        cfg["padding_free"] = False
    if "eval_strategy" not in params and "evaluation_strategy" in params:
        cfg["evaluation_strategy"] = cfg.pop("eval_strategy")
    missing = [k for k in cfg if k not in params and k != "dataset_text_field"]
    assert not missing or any(p.kind == p.VAR_KEYWORD for p in params.values()), "SFTConfig lacks %s: tell Claude" % missing
    args = SFTConfig(**cfg)
    trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=Dataset.from_dict({"text": texts["train"]}),
                         eval_dataset=Dataset.from_dict({"text": texts["val"]}), args=args)
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    longest = max(len(x) for x in trainer.train_dataset["input_ids"])
    assert longest >= texts["max_len"], "examples were cut to %d tokens (longest is %d): tell Claude" % (longest, texts["max_len"])
    labels = trainer.train_dataset[0]["labels"]
    learned = tok.decode([t for t in labels if t != -100])
    assert '"columns"' in learned and "<|im_end|>" in learned, "the answer is not what the model learns: tell Claude"
    assert "Rate these" not in learned and "name | type | missing" not in learned and "<|im_start|>user" not in learned, \
        "the prompt is not masked out: tell Claude"
    settings = {k: getattr(args, k, None) for k in (
        "num_train_epochs", "learning_rate", "lr_scheduler_type", "warmup_steps", "per_device_train_batch_size",
        "gradient_accumulation_steps", "eval_steps", "save_steps", "save_total_limit", "load_best_model_at_end",
        "metric_for_best_model", "greater_is_better", "max_length", "max_seq_length", "padding_free", "seed")}
    settings = {k: (v if isinstance(v, (int, float, str, bool, type(None))) else str(v)) for k, v in settings.items()}
    print("seed %d: about %d updates, %d of them warm-up; validation loss and a checkpoint every %d updates; the model "
          "learns only the answer (%d of %d tokens of the first example)"
          % (seed, updates, warmup, EVAL_STEPS, sum(1 for t in labels if t != -100), len(labels)))
    return trainer, settings

def add_lora(model, seed):
    return FastLanguageModel.get_peft_model(
        model, r=LORA_RANK,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=LORA_RANK, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
        random_state=seed, max_seq_length=MAX_SEQ_LENGTH)

def seed_paths(seed):
    return {"answers": SEED_FILE % seed, "record": SEED_RECORD % seed, "lora": "%s-s%d-lora" % (OUTPUT_NAME, seed),
            "outputs": "outputs_s%d" % seed}

In [ ]:
# ---- three training runs (seeds 3407, 3408, 3409): train, keep the best checkpoint, answer 3 times ----
# (about 2 hours each on an A100; a finished run is kept when this cell runs again)
SEED_RESULTS = {}
for seed in SEEDS:
    p = seed_paths(seed)
    rec = read_json(p["record"])
    if rec and rec.get("session") and complete(p["answers"], K_SAMPLES, rec["session"]):
        print("seed %d: trained and answered earlier (%s): kept" % (seed, rec["session"]))
        SEED_RESULTS[seed] = rec
        continue
    print("\n======== seed %d ========" % seed)
    trained = bool(rec and rec.get("session") and os.path.exists(os.path.join(p["lora"], "adapter_config.json")))
    if trained:   # trained earlier, answers not finished: the saved add-on answers
        model, tokenizer = load_model(p["lora"])
        tok, chat = text_parts(tokenizer)
        check_prompt(chat, "seed %d (reloaded)" % seed)
        print("seed %d: trained earlier (%s); loaded %s to finish its answers" % (seed, rec["session"], p["lora"]))
    else:
        model, tokenizer = load_any()
        tok, chat = text_parts(tokenizer)
        check_prompt(chat, "seed %d" % seed)
        texts = training_texts(tok, chat)
        model = add_lora(model, seed)
        trainer, settings = make_trainer(model, tok, seed, texts, p["outputs"])
        t0 = time.time()
        stats = trainer.train()
        best_dir = trainer.state.best_model_checkpoint
        m = re.search(r"checkpoint-(\d+)", str(best_dir or ""))
        rec = {"seed": seed, "session": "s%d trained %s" % (seed, time.strftime("%Y-%m-%d %H:%M:%S")),
               "train_minutes": round((time.time() - t0) / 60, 1), "train_loss": round(float(stats.training_loss), 4),
               "eval_loss_by_step": [[int(x["step"]), round(float(x["eval_loss"]), 4)]
                                     for x in trainer.state.log_history if "eval_loss" in x],
               "steps": int(trainer.state.global_step),
               "best_checkpoint": {"step": int(m.group(1)) if m else getattr(trainer.state, "best_global_step", None),
                                   "eval_loss": None if trainer.state.best_metric is None
                                   else round(float(trainer.state.best_metric), 4), "dir": best_dir},
               "peak_gpu_gb": round(torch.cuda.max_memory_reserved() / 1024**3, 1), "trainer_settings": settings}
        final = trainer.evaluate().get("eval_loss")   # the loaded model's validation loss: the best checkpoint's
        rec["eval_loss_loaded_model"] = None if final is None else round(float(final), 4)
        best = rec["best_checkpoint"]["eval_loss"]
        if best is None or final is None or abs(float(final) - best) > 0.002:
            print("NOTE: the model in memory may not be the best checkpoint (validation loss %s vs best %s): tell Claude"
                  % (final, best))
        print("seed %d: trained in %.0f minutes (%d steps); validation loss by step %s; best: step %s, %.4f"
              % (seed, rec["train_minutes"], rec["steps"], rec["eval_loss_by_step"], rec["best_checkpoint"]["step"],
                 best if best is not None else float("nan")))
        model.save_pretrained(p["lora"])   # saved now, so a crash while answering cannot lose the training
        tokenizer.save_pretrained(p["lora"])   # with the chat template, whichever object holds it
        if tok is not tokenizer:
            tok.save_pretrained(p["lora"])
        write_json(p["record"], rec)
        del trainer
        free_gpu()
    batch_setting(model, tok, chat)   # the setting the untrained model used
    ans = answer_all(model, tok, chat, out_file=p["answers"], label="trained s%d (run 5)" % seed, session=rec["session"])
    rec["answers"] = stage_stats(ans, "seed %d answers" % seed)
    rec["oom_splits"] = SPLITS["oom_splits"]
    write_json(p["record"], rec)
    SEED_RESULTS[seed] = rec
    del model, tokenizer
    free_gpu()
RUN["seeds"] = {str(s): SEED_RESULTS[s] for s in SEEDS}
write_json(RUN_FILE, RUN)

In [ ]:
# ---- the run record and the fingerprints of the answers so far ----
def answer_files():
    # the untrained and the three trained models' files, and each GGUF file's once it is complete
    fs = [BASE_FILE] + [SEED_FILE % s for s in SEEDS]
    fs += [GGUF_FILE % q for q in GGUF_QUANTS if RUN.get("gguf", {}).get(q, {}).get("answered")]
    return [f for f in fs if os.path.exists(f)]

def write_manifest():
    for f in [BASE_FILE] + [SEED_FILE % s for s in SEEDS]:
        verify_file(f, K_SAMPLES)
    for f in answer_files()[1 + len(SEEDS):]:
        verify_file(f, 1)
    RUN["answer_files"] = {f: sha256(f) for f in answer_files()}
    RUN["settings"] = {"max_seq_length": MAX_SEQ_LENGTH, "train_max_tokens": TRAIN_MAX_TOKENS, "epochs": EPOCHS,
                       "learning_rate": LEARNING_RATE, "lr_scheduler": LR_SCHEDULER, "warmup_ratio": WARMUP_RATIO,
                       "lora_rank": LORA_RANK, "lora_alpha": LORA_RANK, "batch_size": BATCH_SIZE,
                       "grad_accum": GRAD_ACCUM, "eval_steps": EVAL_STEPS, "save_total_limit": SAVE_TOTAL_LIMIT,
                       "best_model_metric": "eval_loss", "seeds": SEEDS,
                       "eval": {"k_samples": K_SAMPLES, "answer_batch": ANSWER_BATCH, "gen_seed": GEN_SEED,
                                "max_new_tokens": EVAL_MAX_NEW_TOKENS, "temperature": EVAL_TEMPERATURE,
                                "top_p": EVAL_TOP_P, "top_k": EVAL_TOP_K, "thinking": False,
                                "retry": "once, if a column is unrated by every readable sample or no sample is readable",
                                "gguf": {"quants": GGUF_QUANTS, "engine": "llama.cpp llama-server", "greedy": True,
                                         "answers_per_prompt": 1}},
                       "training_text": "test prompt (thinking off) + answer json + <|im_end|>"}
    write_json(RUN_FILE, RUN)
    man = {"run": 5, "created": time.strftime("%Y-%m-%d %H:%M:%S"), "inputs": RUN["files"],
           "answers": RUN["answer_files"], "run_record": {RUN_FILE: sha256(RUN_FILE)},
           "seed_records": {SEED_RECORD % s: sha256(SEED_RECORD % s) for s in SEEDS if os.path.exists(SEED_RECORD % s)},
           "note": "SHA-256 of every answer file of run 5, made before anyone sees test 5's answer key"}
    write_json(MANIFEST_FILE, man)
    keep = answer_files() + [RUN_FILE, MANIFEST_FILE, CHECK_FILE] + \
        [SEED_RECORD % s for s in SEEDS if os.path.exists(SEED_RECORD % s)]
    tmp = RESULTS_ZIP + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in keep:
            if os.path.exists(f):
                z.write(f)
    os.replace(tmp, RESULTS_ZIP)
    print("%s and %s written; %s holds %d files (%.0f MB)"
          % (RUN_FILE, MANIFEST_FILE, RESULTS_ZIP, len(keep), os.path.getsize(RESULTS_ZIP) / 1e6))
    for f, h in RUN["answer_files"].items():
        print("  %s  %s" % (h, f))

write_manifest()
print("The main answers are done. You could stop here and download %s; the cells below add the two laptop-size "
      "models (GGUF), about 1.5 hours more." % RESULTS_ZIP)

In [ ]:
# ---- the laptop files: the run with the lowest validation loss, merged and saved as Q4_K_M and Q6_K GGUF ----
# (20-30 minutes; the files go in this folder)
best_seed = min(SEEDS, key=lambda s: (SEED_RESULTS[s]["best_checkpoint"]["eval_loss"]
                                      if SEED_RESULTS[s]["best_checkpoint"]["eval_loss"] is not None else 9e9, s))
RUN["gguf_seed"] = best_seed
GGUF_PATHS = {q: "%s-s%d-%s.gguf" % (OUTPUT_NAME, best_seed, q.upper()) for q in GGUF_QUANTS}
print("lowest validation loss: seed %d (%s)" % (best_seed, ", ".join(
    "s%d %.4f" % (s, SEED_RESULTS[s]["best_checkpoint"]["eval_loss"] or float("nan")) for s in SEEDS)))
model, tokenizer = load_model(seed_paths(best_seed)["lora"])
tok, chat = text_parts(tokenizer)
check_prompt(chat, "seed %d (reloaded for the GGUF files)" % best_seed)
PROMPTS = {r["id"]: prompt_text(chat, r["system"], r["user"]) for r in TEST_ROWS}       # the exact prompt texts
RETRY_PROMPTS = lambda r, rem: prompt_text(chat, r["system"], r["user"] + "\n\n" + rem)  # noqa: E731
try:
    if all(os.path.exists(p) for p in GGUF_PATHS.values()):
        print("GGUF files already here: kept")
    else:
        t0 = time.time()
        model.save_pretrained_gguf(OUTPUT_NAME + "-s%d" % best_seed, tok, quantization_method=GGUF_QUANTS)
        found = [p for p in glob.glob("**/*.gguf", recursive=True)
                 if os.path.getmtime(p) >= t0 - 5 and "mmproj" not in os.path.basename(p).lower()]
        for p in found:
            print("  written: %s (%.1f GB)" % (p, os.path.getsize(p) / 1024**3))
        for q, final in GGUF_PATHS.items():
            named = [p for p in found if q.lower() in os.path.basename(p).lower()]
            assert named, "no %s .gguf file was written" % q
            src = max(named, key=os.path.getsize)
            if os.path.abspath(src) != os.path.abspath(final):
                os.replace(src, final)
        RUN["minutes"]["gguf export"] = round((time.time() - t0) / 60, 1)
    prev = RUN.get("gguf") or {}
    RUN["gguf"] = {}
    for q, p in GGUF_PATHS.items():
        h = sha256(p)
        same = (prev.get(q) or {}).get("sha256") == h   # answers count only for the very file they were made with
        RUN["gguf"][q] = {"file": p, "gb": round(os.path.getsize(p) / 1024**3, 2), "sha256": h,
                          "answered": bool(same and (prev.get(q) or {}).get("answered"))}
    for q, d in RUN["gguf"].items():
        print("%s: %s (%.1f GB)" % (q, d["file"], d["gb"]))
except Exception as e:
    RUN["gguf"] = {}
    RUN["gguf_error"] = "%s: %s" % (e.__class__.__name__, " ".join(str(e).split())[:300])
    print("The GGUF files could not be made (%s). The laptop answers are left out; everything else is done: download "
          "%s and tell Claude." % (RUN["gguf_error"], RESULTS_ZIP))
del model
free_gpu()
write_json(RUN_FILE, RUN)

In [ ]:
# ---- llama.cpp's server, built for this GPU (10-20 minutes the first time) ----
def llama_dirs():
    return [d for d in ("llama.cpp", os.path.expanduser("~/.unsloth/llama.cpp"), "/workspace/llama.cpp")
            if os.path.isfile(os.path.join(d, "CMakeLists.txt"))]

def build_llama_server():
    for d in llama_dirs():
        exe = os.path.join(d, "build-server", "bin", "llama-server")
        if os.path.exists(exe):
            return exe
    if shutil.which("cmake") is None:
        pip("cmake")
    if not llama_dirs():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "llama.cpp"])
    d = llama_dirs()[0]
    b = os.path.join(d, "build-server")
    base = ["cmake", "-S", d, "-B", b, "-DGGML_CUDA=ON", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"]
    try:
        subprocess.check_call(base + ["-DCMAKE_CUDA_ARCHITECTURES=native"])
    except subprocess.CalledProcessError:
        shutil.rmtree(b, ignore_errors=True)
        subprocess.check_call(base)
    subprocess.check_call(["cmake", "--build", b, "--config", "Release", "-j", str(os.cpu_count() or 8),
                           "--target", "llama-server"])
    return os.path.join(b, "bin", "llama-server")

t0 = time.time()
try:
    assert RUN.get("gguf"), "no GGUF files to answer with"
    LLAMA_SERVER = build_llama_server()
    RUN["llama_server"] = {"binary": LLAMA_SERVER, "minutes": round((time.time() - t0) / 60, 1)}
    try:
        RUN["llama_server"]["version"] = subprocess.run([LLAMA_SERVER, "--version"], capture_output=True, text=True,
                                                        timeout=60).stderr.strip()[-300:]
    except Exception:
        pass
    print("llama-server: %s (%.0f min)" % (LLAMA_SERVER, (time.time() - t0) / 60))
except Exception as e:
    LLAMA_SERVER = None
    RUN["llama_server"] = {"error": "%s: %s" % (e.__class__.__name__, " ".join(str(e).split())[:300])}
    print("llama.cpp could not be built (%s). The laptop answers are left out; everything else is done: download %s "
          "and tell Claude." % (RUN["llama_server"]["error"], RESULTS_ZIP))
write_json(RUN_FILE, RUN)

In [ ]:
# ---- each GGUF file answers every prompt once, greedy, with llama.cpp (about 20 minutes each) ----
import urllib.request
from concurrent.futures import ThreadPoolExecutor

def http(path, body=None, timeout=900):
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (LLAMA_PORT, path),
                                 data=None if body is None else json.dumps(body).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def start_server(gguf):
    log = open("llama_server_%s.log" % os.path.basename(gguf), "w")
    proc = subprocess.Popen([LLAMA_SERVER, "-m", gguf, "-c", str(LLAMA_SLOTS * LLAMA_CTX_PER_SLOT),
                             "-np", str(LLAMA_SLOTS), "-ngl", "999", "--host", "127.0.0.1", "--port", str(LLAMA_PORT)],
                            stdout=log, stderr=subprocess.STDOUT)
    for _ in range(600):
        time.sleep(1)
        if proc.poll() is not None:
            raise RuntimeError("llama-server stopped (see %s)" % log.name)
        try:
            if http("/health", timeout=5).get("status") == "ok":
                return proc
        except Exception:
            pass
    proc.kill()
    raise RuntimeError("llama-server did not start in 10 minutes (see %s)" % log.name)

def complete_text(text):
    r = http("/completion", {"prompt": text, "n_predict": EVAL_MAX_NEW_TOKENS, "temperature": 0.0, "seed": GEN_SEED,
                             "cache_prompt": False, "stop": ["<|im_end|>", "<|endoftext|>"]})
    return r.get("content") or "", int(r.get("tokens_predicted") or 0)

def gguf_answers(q, gguf):
    out_file = GGUF_FILE % q
    session = "gguf %s s%d %s" % (q, RUN["gguf_seed"], RUN["gguf"][q]["sha256"][:12])
    done = load_answers(out_file, session)
    todo = [r for r in TEST_ROWS if r["id"] not in done]
    if not todo:
        return done
    proc = start_server(gguf)
    try:
        toks = http("/tokenize", {"content": "<|im_start|>user"}).get("tokens") or []
        assert len(toks) <= 3, "llama-server does not read the chat's special tokens (%s): tell Claude" % toks
        t0 = time.time()

        def one(r):
            s0 = time.time()
            text, n = complete_text(PROMPTS[r["id"]])
            got = rated_cols(risks_of(read_answer(text), r["columns"]))
            rec = {"id": r["id"], "table": r["table"], "group": r["group"], "condition": r["condition"],
                   "label": "gguf %s (seed %d)" % (q, RUN["gguf_seed"]), "thinking": False, "texts": [text],
                   "texts_retry": None, "tokens": [n], "seconds": None, "attempts": 1, "session": session,
                   "readable": int(bool(got)), "batch": LLAMA_SLOTS}
            missing = [c for c in r["columns"] if c not in got]
            if not got or missing:
                names = [" ".join(str(c).split()) for c in missing]
                rem = REMINDER_ALL % len(r["columns"]) if not got else REMINDER_MISSING % ", ".join(names)
                t2, n2 = complete_text(RETRY_PROMPTS(r, rem))
                rec.update(texts_retry=[t2], tokens=[n, n2], attempts=2)
            rec["seconds"] = round(time.time() - s0, 1)
            return rec

        with ThreadPoolExecutor(max_workers=LLAMA_SLOTS) as pool:
            for k, rec in enumerate(pool.map(one, todo), 1):
                done[rec["id"]] = rec
                if k % 24 == 0 or k == len(todo):
                    write_all(out_file, done)
                    el = time.time() - t0
                    print("  gguf %s: %d / %d prompts (%.0f min, about %.0f min left)"
                          % (q, len(done), len(TEST_ROWS), el / 60, el / k * (len(todo) - k) / 60))
        write_all(out_file, done)
    finally:
        proc.terminate()
        try:
            proc.wait(timeout=60)
        except Exception:
            proc.kill()
    verify_file(out_file, 1)
    return done

if LLAMA_SERVER:
    for q, d in RUN["gguf"].items():
        try:
            ans = gguf_answers(q, d["file"])
            d["answered"] = True
            d["answers"] = stage_stats(ans, "gguf %s answers" % q)
        except Exception as e:
            d["error"] = "%s: %s" % (e.__class__.__name__, " ".join(str(e).split())[:300])
            print("gguf %s: %s -- left out; tell Claude" % (q, d["error"]))
        write_json(RUN_FILE, RUN)

In [ ]:
# ---- the final run record, the fingerprints of every answer file, and run5_results.zip ----
RUN["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
write_manifest()
print("\nDone. Download %s (right-click > Download)%s, then terminate the pod in the RunPod console."
      % (RESULTS_ZIP, " and, for LM Studio, %s" % RUN["gguf"]["q4_k_m"]["file"] if RUN.get("gguf") else ""))

## Done

1. In the file list on the left, right-click **`run5_results.zip`** > **Download** (it holds every answer file, the run
   record and the fingerprints).
2. Optional, for LM Studio: right-click the `...-Q4_K_M.gguf` file > **Download** (about 5-6 GB).
3. In the RunPod console, open **Pods** and **terminate** this pod. Billing stops only then.
4. Back on your Mac: GUIDE.md, Part B step 5 (t5_import_gpu.py).